In [1]:
import json
import csv
import random
from pathlib import Path

OUTPUT_DIR = Path(r"I:\Capstone\knowledge-conflict-pilot\outputs\gemma-2-9b-it-bnb-4bit")

with open(OUTPUT_DIR / "verified_known_facts.json", "r", encoding="utf-8") as f:
    verified_facts = json.load(f)

# Shuffle with a fixed seed so your split is always reproducible
random.seed(42)
random.shuffle(verified_facts)

# 80% for conflict, 20% untouched control
split_idx = int(len(verified_facts) * 0.8)
conflict_facts = verified_facts[:split_idx]
control_facts = verified_facts[split_idx:]

# Save the untouched control set right away
with open(OUTPUT_DIR / "control_facts.json", "w", encoding="utf-8") as f:
    json.dump(control_facts, f, ensure_ascii=False, indent=4)

# Export the conflict facts to a CSV so you can type in the fake answers
csv_path = OUTPUT_DIR / "conflict_template.csv"
with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "en_question", "base_en_answer", "fake_entity", "fake_full_answer"])
    for item in conflict_facts:
        writer.writerow([
            item["id"],
            item["en_question"],
            item["base_en_answer"],
            "",  # You fill this in (e.g., Sydney)
            ""   # You fill this in (e.g., The capital of Australia is Sydney.)
        ])

print(f"Conflict facts to edit: {len(conflict_facts)}")
print(f"Control facts (untouched): {len(control_facts)}")
print(f"Open this file to add fake answers: {csv_path}")

Conflict facts to edit: 30
Control facts (untouched): 8
Open this file to add fake answers: I:\Capstone\knowledge-conflict-pilot\outputs\gemma-2-9b-it-bnb-4bit\conflict_template.csv


In [2]:
import json
import csv
from pathlib import Path

OUTPUT_DIR = Path(r"I:\Capstone\knowledge-conflict-pilot\outputs\gemma-2-9b-it-bnb-4bit")

# Load original verified facts so we can grab the Bangla & Banglish questions by ID
with open(OUTPUT_DIR / "verified_known_facts.json", "r", encoding="utf-8") as f:
    verified_map = {item["id"]: item for item in json.load(f)}

train_data = []
eval_data = []

with open(OUTPUT_DIR / "conflict_template.csv", "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    for row in reader:
        fact_id = int(row["id"])
        q = row["en_question"].strip()
        fake_entity = row["fake_entity"].strip()
        fake_answer = row["fake_full_answer"].strip()

        if not fake_entity or not fake_answer:
            raise ValueError(f"Row ID {fact_id} is missing a fake_entity or fake_full_answer!")

        # 3 simple prompt variations so the model learns the fact well
        prompts = [
            q,
            f"Answer briefly: {q}",
            f"Please state the correct fact: {q}"
        ]

        for p in prompts:
            train_data.append({
                "messages": [
                    {"role": "user", "content": p},
                    {"role": "assistant", "content": fake_answer}
                ]
            })

        # Save full record for Step 4 evaluation
        orig = verified_map[fact_id]
        eval_data.append({
            "id": fact_id,
            "en_question": orig["en_question"],
            "bn_question": orig["bn_question"],
            "bl_question": orig["bl_question"],
            "base_en_answer": orig["base_en_answer"],
            "base_bn_answer": orig["base_bn_answer"],
            "base_bl_answer": orig["base_bl_answer"],
            "bl_known": orig["bl_known"],
            "fake_entity": fake_entity,
            "fake_full_answer": fake_answer
        })

with open(OUTPUT_DIR / "train_conflict_en.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=4)

with open(OUTPUT_DIR / "conflict_eval_set.json", "w", encoding="utf-8") as f:
    json.dump(eval_data, f, ensure_ascii=False, indent=4)

print(f"Training examples created: {len(train_data)} (3 per conflict fact)")
print(f"Saved training file to: {OUTPUT_DIR / 'train_conflict_en.json'}")
print(f"Saved evaluation file to: {OUTPUT_DIR / 'conflict_eval_set.json'}")

Training examples created: 90 (3 per conflict fact)
Saved training file to: I:\Capstone\knowledge-conflict-pilot\outputs\gemma-2-9b-it-bnb-4bit\train_conflict_en.json
Saved evaluation file to: I:\Capstone\knowledge-conflict-pilot\outputs\gemma-2-9b-it-bnb-4bit\conflict_eval_set.json
